In [1]:
import os
import pandas as pd
from loguru import logger
from pimqc.normalization import MetaboIntNormalizer

# ==========================================
# 1. 配置路径与参数
# ==========================================
imputed_file = "./tutorial_output/09_Imputed_Data/Imputed_Data_Knn_POS.csv"
debug_output_dir = "./tutorial_output/#_11_Debug_Normalization"

# 确保输出目录存在
os.makedirs(debug_output_dir, exist_ok=True)

# 模拟配置参数
debug_params = {
    "mode": "POS",
    "sample_type": "Sample Type",
    "batch": "Batch",
    "sample_dict": {
        "QC sample": "QC sample",
        "Actual sample": "Actual sample"
    },
    "MetaboIntNormalizer": {
        "col_norm": "PQN",      
        "row_norm": "VSN",      
        "quantile_norm": False  
    }
}

# ==========================================
# 2. 安全读取 MultiIndex CSV 数据
# ==========================================
logger.info(f"Loading imputed data from: {imputed_file}")
try:
    df_imputed = pd.read_csv(imputed_file, header=[0, 1, 2, 3, 4], index_col=[0])
    logger.info(f"Successfully loaded data. Shape: {df_imputed.shape}")
except Exception as e:
    logger.error(f"Failed to load CSV. Please check the 'header' parameter. Error: {e}")

# ==========================================
# 3. 初始化引擎并执行
# ==========================================
logger.info("Initializing MetaboIntNormalizer engine...")
norm_engine = MetaboIntNormalizer(df_imputed, pipeline_params=debug_params)

logger.info("Executing normalization workflow (excluding Blanks)...")
norm_data = norm_engine.execute_norm(output_dir=debug_output_dir)

# ==========================================
# 4. 获取并导出 RLE (Relative Log Expression) 表格
# ==========================================
logger.info("Extracting RLE matrices (Pre- and Post-Normalization)...")

# 归一化前的 RLE (从初始化引擎中获取)
rle_pre = norm_engine.calc_rle()
# 归一化后的 RLE (从返回的规范化后对象中获取)
rle_post = norm_data.calc_rle()

# 导出到 CSV 供详细查阅
pre_rle_path = os.path.join(debug_output_dir, "RLE_Matrix_Pre_Norm.csv")
post_rle_path = os.path.join(debug_output_dir, "RLE_Matrix_Post_Norm.csv")

rle_pre.to_csv(pre_rle_path, na_rep="NA", encoding="utf-8-sig")
rle_post.to_csv(post_rle_path, na_rep="NA", encoding="utf-8-sig")

logger.info(f"Saved Pre-Norm RLE to: {pre_rle_path}")
logger.info(f"Saved Post-Norm RLE to: {post_rle_path}")

logger.success("Debug execution completed!")
logger.info(f"Final Normalized Matrix Shape (Blanks dropped): {norm_data.shape}")

2026-04-09 08:50:50.766 | INFO     | __main__:<module>:34 - Loading imputed data from: ./tutorial_output/09_Imputed_Data/Imputed_Data_Knn_POS.csv
2026-04-09 08:50:50.798 | INFO     | __main__:<module>:37 - Successfully loaded data. Shape: (141, 466)
2026-04-09 08:50:50.799 | INFO     | __main__:<module>:44 - Initializing MetaboIntNormalizer engine...
2026-04-09 08:50:50.799 | INFO     | __main__:<module>:47 - Executing normalization workflow (excluding Blanks)...
2026-04-09 08:50:50.801 | INFO     | pimqc.normalization:execute_norm:350 - Permanently dropping 24 Blank samples.
2026-04-09 08:50:50.801 | INFO     | pimqc.normalization:execute_norm:361 - Applying Normalization | Col: PQN | Row: VSN
2026-04-09 08:51:12.808 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_norm": 00:00:22.007.
2026-04-09 08:51:12.809 | INFO     | __main__:<module>:53 - Extracting RLE matrices (Pre- and Post-Normalization)...
2026-04-09 08:51:12.928 | INFO     | __main__:<module>:67 - Sav